# Evaluating Robustness: Noise Sensitivity in Advanced RAG Systems

In Retrieval-Augmented Generation (RAG), the quality of the final answer hinges not only on retrieving relevant documents but also on ensuring that the model does not become distracted or misled by extraneous information within those retrieved chunks. This phenomenon is known as "noise sensitivity." A highly noise-sensitive system might incorporate minor, irrelevant details from a noisy context into its response, even if those details contradict or dilute the core truth established in the reference material.

This notebook introduces and demonstrates the use of the `NoiseSensitivity` metric from Ragas. This advanced evaluation technique measures how well a generated response remains grounded in the provided *reference* (the ground truth) despite the presence of potentially misleading, irrelevant, or overly detailed information within the broader set of *retrieved contexts*. For developers building production-grade RAG pipelines, understanding and mitigating noise sensitivity is crucial for achieving reliable, trustworthy outputs. A low score indicates that the model successfully filtered out the noise and focused only on the core facts.

For those working with complex orchestration frameworks like LangGraph, robust evaluation metrics such as Noise Sensitivity are vital components of the validation loop. Before committing an answer to a user, advanced systems must pass rigorous checks—including grounding against known references and ignoring irrelevant context—to prevent subtle hallucinations or over-generalization. By mastering this metric, you learn how to build RAG pipelines that are not just accurate, but demonstrably robust against real-world data noise.

### Learning Objectives

Upon completing this notebook, you will be able to:

*   **Define Noise Sensitivity:** Explain the concept of noise sensitivity in the context of RAG and its implications for model reliability.
*   **Utilize `NoiseSensitivity`:** Implement and interpret the `NoiseSensitivity` metric using the `ragas` library.
*   **Analyze Grounding Failures:** Differentiate between a response that is merely *plausible* (Example 1) versus one that is strictly *grounded* in the reference material (Example 2).
*   **Improve RAG Robustness:** Understand how evaluating noise sensitivity helps identify weaknesses in both the retrieval component and the generation model, leading to more robust production systems.


### Noise Sensitivity Scorer Initialization

This cell initializes the `NoiseSensitivity` metric scorer from Ragas. It sets up an asynchronous OpenAI client and uses a factory function to instantiate a specific LLM (`gpt-5-mini`) which is then passed to the `NoiseSensitivity` class, making it ready to calculate how robust the generated answers are to minor input perturbations.


In [ ]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import NoiseSensitivity

# Initialize the asynchronous OpenAI client for API calls
client = AsyncOpenAI()
# Use the factory to get an LLM instance (e.g., gpt-5-mini) using the configured client
llm = llm_factory("gpt-5-mini", client=client)

# Initialize the NoiseSensitivity scorer, passing the required LLM object
scorer = NoiseSensitivity(llm=llm)


d:\rag-evaluation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


This cell demonstrates the 'Noise Sensitivity' scoring mechanism. It evaluates how much a generated response relies on potentially misleading or tangential information (noise) present in the retrieved context, even if the core answer is mostly correct.


In [2]:
# Example 1: Response is mostly correct but 'contributes to financial stability'
# is a claim drawn from the noisy context rather than the core reference
result = await scorer.ascore(
    user_input="What is the Life Insurance Corporation of India (LIC) known for?",
    response="The Life Insurance Corporation of India (LIC) is the largest insurance company in India, known for its vast portfolio of investments. LIC contributes to the financial stability of the country.",
    reference="The Life Insurance Corporation of India (LIC) is the largest insurance company in India, established in 1956 through the nationalization of the insurance industry. It is known for managing a large portfolio of investments.",
    retrieved_contexts=[
        "The Life Insurance Corporation of India (LIC) was established in 1956 following the nationalization of the insurance industry in India.",
        "LIC is the largest insurance company in India, with a vast network of policyholders and huge investments.",
        "As the largest institutional investor in India, LIC manages substantial funds, contributing to the financial stability of the country.",
        "The Indian economy is one of the fastest-growing major economies in the world, thanks to sectors like finance, technology, and manufacturing."
    ]
)
# Call the ascore method (assuming 'scorer' is an initialized scoring object).
# This calculates the noise sensitivity score based on the inputs.
print(f"Noise Sensitivity Score: {result.value}")



Noise Sensitivity Score: 0.3333333333333333


This cell demonstrates the 'Noise Sensitivity' scoring capability, which measures how well a generated response remains grounded in the core facts even when presented with irrelevant or noisy context documents. Here, we test if the model ignores distracting information (like the Louvre or French cuisine) and focuses only on the direct answer.


In [3]:
# Example 2 : Response stays grounded in the reference despite noisy contexts
# We use the scorer to evaluate how well the response adheres to the core facts.
result = await scorer.ascore(
    user_input="Where is the Eiffel Tower located?",
    response="The Eiffel Tower is located in Paris, France.",
    reference="The Eiffel Tower is located in Paris, France.",
    retrieved_contexts=[
        "The Eiffel Tower is a landmark located in Paris, France.", # Context 1: Relevant information
        "Paris is also home to the Louvre Museum, which contains thousands of artworks including the Mona Lisa.", # Context 2: Noise/Irrelevant context
        "France is known for its cuisine, wine, and fashion industry."
    ]
)
# Print the calculated score, indicating the degree of noise sensitivity.
print(f"Noise Sensitivity Score: {result.value}")


Noise Sensitivity Score: 0.0
